### INSTRUCTIONS

For this assignment, you MUST use the AWS VM that has been assigned to you.

First, review the Image Segmentation lecture slides.

Then, read through the entire framework below once. Note the parts you need to code up.

NOTE: YOU WILL ONLY EDIT ONE CELL (TASKS 1.1 - 1.7) FOR THIS ASSIGNMENT AS PER THE INSTRUCTIONS IN THAT CELL

In [9]:
##### DO NOT EDIT THIS CELL!!!
##### DO NOT IMPORT ANY ADDITIONAL PACKAGES

import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
%matplotlib inline

from tensorflow.keras.layers.experimental import preprocessing 
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import applications

ModuleNotFoundError: No module named 'tensorflow.keras.layers.experimental'

In [10]:
# https://www.tensorflow.org/datasets/catalog/oxford_iiit_pet
ds_name = 'oxford_iiit_pet:3.*.*' # 'oxford_iiit_pet'
input_shape = (None, None, 3)

In [11]:
##### DO NOT EDIT THIS CELL!!!

# load dataset
# code adapted from https://www.tensorflow.org/tutorials/images/segmentation

(train_ds, val_ds, test_ds), metadata = tfds.load(
    # more datasets: https://www.tensorflow.org/datasets/catalog/overview#all_datasets
    name=ds_name,
    # more info: https://www.tensorflow.org/datasets/splits
    split=['train', 'test[:75%]', 'test[75%:]'], 
    # return metadata
    with_info=True, 
    # download data
    download=True, 
    # whether to shuffle the files between each epoch
    shuffle_files=False, 
)

print (metadata.features)

# num classes
num_classes = metadata.features['label'].num_classes
print (num_classes)

# class labels
class_names = metadata.features['label'].names
print (class_names)

plt.figure(figsize=(15, 15))
i = 0
for datapoint in train_ds.take(3):
    label = datapoint['label']
    image = datapoint['image']
    mask = datapoint['segmentation_mask']
    ax = plt.subplot(3, 2, i + 1)
    plt.title(class_names[label])
    plt.imshow(image.numpy().astype("uint8"))
    ax = plt.subplot(3, 2, i + 2)
    plt.title('mask')
    plt.imshow(mask.numpy().astype("uint8"))
    plt.axis("off")
    i = i + 2
    
# get shape of images and masks 
iter(train_ds).get_next()['image'].shape, iter(train_ds).get_next()['segmentation_mask'].shape

2025-04-22 14:16:24.025906: W external/local_xla/xla/tsl/platform/cloud/google_auth_provider.cc:184] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Could not resolve hostname', error details: Could not resolve host: metadata.google.internal".


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

KeyboardInterrupt: 

In [ ]:
##### DO NOT EDIT THIS CELL!!!

# configure dataset for performance
# code adapted from https://www.tensorflow.org/guide/data_performance

IMG_SIZE = 128
batch_size = 64
BUFFER_SIZE = 1000
AUTOTUNE = tf.data.experimental.AUTOTUNE

def prepare(ds, shuffle=False, resize=False, normalize=False, augment=False):
    
    # Extract just image and segmentation_mask
    ds = ds.map(lambda x: (x['image'], x['segmentation_mask']))
    
    # Resize dataset 
    if resize:
        resizing = preprocessing.Resizing(IMG_SIZE,IMG_SIZE)
        ds = ds.map(lambda image, mask: (resizing(image), resizing(mask)))

    # Augment dataset (should only be applied to training data)
    if augment:
        ds = ds.map(lambda image, mask: (tf.image.flip_left_right(image), tf.image.flip_left_right(mask)) if tf.random.uniform(()) > 0.5 else (image, mask))
        
    # Normalize dataset 
    if normalize:
        # image will now have values 0-1 (instead of 0-255)
        # mask will now have values 0, 1, 2 (instead of 1, 2, 3)
        ds = ds.map(lambda image, mask: (image/255.0, mask-1)) 
    
    # Batch dataset
    # Note: To be batched, images need to share the same height and width.
    #          So if dataset has images of different sizes, resize them before batching
    ds = ds.batch(batch_size)
    
    # The first time the dataset is iterated over, its elements will be cached either in the specified file or in memory;
    #   subsequent iterations will use the cached data.
    #   Note: cache will produce exactly the same elements during each iteration through the dataset;
    #     if you wish to randomize the iteration order, make sure to call shuffle after calling cache.
    ds = ds.cache()
    
    # Shuffle the data. 
    #   buffer_size: this dataset fills a buffer with buffer_size elements, then randomly samples elements from this buffer, 
    #     replacing the selected elements with new elements. For perfect shuffling, a buffer size greater than or equal to 
    #     the full size of the dataset is required.
    if shuffle:
        ds = ds.shuffle(BUFFER_SIZE, seed=None)

    # Prefetching overlaps the preprocessing and model execution of a training step. 
    #   While the model is executing training step s, the input pipeline is reading the data for step s+1
    ds = ds.prefetch(buffer_size=AUTOTUNE)
        
    return ds

train_ds = prepare(train_ds, shuffle=True, resize=True, normalize=True, augment=True)
val_ds = prepare(val_ds, resize=True, normalize=True)
test_ds = prepare(test_ds, resize=True, normalize=True)

# get shape of images and masks 
iter(train_ds).get_next()[0].shape, iter(train_ds).get_next()[1].shape

In [ ]:
##### THIS IS THE ONLY CELL YOU WILL EDIT PER THE ISNTRUCTIONS BELOW

### downsampling path - VGG16 
backbone = applications.VGG16(input_shape=(128, 128, 3), include_top=False)
backbone.trainable = False
down1 = backbone.get_layer('block1_pool').output
down2 = backbone.get_layer('block2_pool').output
down3 = backbone.get_layer('block3_pool').output
down4 = backbone.get_layer('block4_pool').output
down5 = backbone.get_layer('block5_pool').output

### use the functional API to create the upsampling path
### NOTE: you will need to pass the output of each step as input to the next step
#  upsampling block 1
#   - transpose convolution with 512 filters, 3x3 kernel, stride of 2, same padding, relu activation
#      (input to this ranspose convolution layer is down5)
#   - batch norm (with default parameters)
#   - concatenate with down4
### (1.1) YOUR CODE HERE
up1 = layers.Conv2DTranspose(512, (3, 3), strides=2, padding='same', activation='relu')(down5)
up1 = layers.BatchNormalization()(up1)
up1 = layers.Concatenate()([up1, down4])

#  upsampling block 2
#   - transpose convolution with 256 filters, 3x3 kernel, stride of 2, same padding, relu activation 
#   - batch norm (with default parameters)
#   - concatenate with down3
### (1.2) YOUR CODE HERE
up2 = layers.Conv2DTranspose(256, (3, 3), strides=2, padding='same', activation='relu')(up1)
up2 = layers.BatchNormalization()(up2)
up2 = layers.Concatenate()([up2, down3])


#  upsampling block 3
#   - transpose convolution with 128 filters, 3x3 kernel, stride of 2, same padding, relu activation 
#   - batch norm (with default parameters)
#   - concatenate with down2
### (1.3) YOUR CODE HERE
up3 = layers.Conv2DTranspose(128, (3, 3), strides=2, padding='same', activation='relu')(up2)
up3 = layers.BatchNormalization()(up3)
up3 = layers.Concatenate()([up3, down2])

#  upsampling block 4
#   - transpose convolution with 64 filters, 3x3 kernel, stride of 2, same padding, relu activation 
#   - batch norm (with default parameters)
#   - concatenate with down1
### (1.4) YOUR CODE HERE
up4 = layers.Conv2DTranspose(64, (3, 3), strides=2, padding='same', activation='relu')(up3)
up4 = layers.BatchNormalization()(up4)
up4 = layers.Concatenate()([up4, down1])


#  upsampling block 5
#   - transpose convolution with 3 filters, 3x3 kernel, stride of 2, same padding, relu activation 
### (1.5) YOUR CODE HERE
up5 = layers.Conv2DTranspose(3, (3, 3), strides=2, padding='same', activation='relu')(up4)


# create model:
#  - set inputs to 'backbone.input'
#  - set outputs to your U-Net output
### (1.6) YOUR CODE HERE
model = models.Model(inputs=backbone.input, outputs=up5)

# compile model
#  - set optimizer to adam (with default learning rate)
#  - set loss to tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
#    NOTE: setting from_logits=True because the output layer of UNet did not have softmax
#  - set metrics to accuracy
### (1.7) YOUR CODE HERE
model.compile(
    optimizer='adam', 
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
    metrics=['accuracy']
)


# print model summary
# you should get: 
#  - Total params: 20,178,819
#  - Trainable params: 5,462,211
#  - Non-trainable params: 14,716,608
model.summary()

In [ ]:
%%time

##### DO NOT EDIT THIS CELL!!!

# train the model
epochs = 7
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

In [ ]:
##### DO NOT EDIT THIS CELL!!!

# model.fit() returns a History object that contains a dictionary with everything that happened during training
# we can use it to plot training/validation loss and training/validation accuracy
history_dict = history.history
print (history_dict.keys())

# plot total loss
loss = history_dict['loss']
val_loss = history_dict['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'b', label='training loss')
plt.plot(epochs, val_loss, 'r', label='validation loss')
plt.title('Training and Validation loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# plot accuracy for clshead
acc = history_dict['accuracy']
val_acc = history_dict['val_accuracy']
epochs = range(1, len(acc) + 1)
plt.plot(epochs, acc, 'b', label='training accuracy')
plt.plot(epochs, val_acc, 'r', label='validation accuracy')
plt.title('Training and Validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
##### DO NOT EDIT THIS CELL!!!

# make predictions
# code adapted from https://www.tensorflow.org/tutorials/images/segmentation

plt.figure(figsize=(15, 15))
i = 0
for image, mask in test_ds.take(3):
    
    ax = plt.subplot(3, 3, i + 1)
    plt.title('image')
    plt.imshow(tf.keras.preprocessing.image.array_to_img(image[0]))
    
    ax = plt.subplot(3, 3, i + 2)
    plt.title('mask')
    plt.imshow(tf.keras.preprocessing.image.array_to_img(mask[0]))
    
    ax = plt.subplot(3, 3, i + 3)
    plt.title('pred_mask')
    pred_mask = model.predict(image)
    pred_mask = tf.argmax(pred_mask, axis=-1)
    pred_mask = pred_mask[..., tf.newaxis]
    plt.imshow(tf.keras.preprocessing.image.array_to_img(pred_mask[0]))
    
    plt.axis("off")
    i = i + 3